# Decision Tree for Classification and Regression.

Step -1- Building a Custom Decision Tree with Information Gain:

In [71]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error

In [72]:
import numpy as np

class CustomDecisionTree:
    def __init__(self, max_depth=None):
        """
        Initialize the decision tree.
        Parameters:
        max_depth: int or None
            Maximum depth of the tree.
        """
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        """
        Train the decision tree on training data.
        """
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        """
        Recursively build the tree based on information gain.
        """
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        # Stopping condition 1: all samples are of same class
        if len(unique_classes) == 1:
            return {'class': unique_classes[0]}

        # Stopping condition 2: reached maximum depth
        if num_samples == 0 or (self.max_depth and depth >= self.max_depth):
            return {'class': np.bincount(y).argmax()}

        best_info_gain = -float('inf')
        best_split = None

        # Iterate over all features and thresholds
        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                left_y = y[left_mask]
                right_y = y[right_mask]

                info_gain = self._information_gain(y, left_y, right_y)
                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }

        if best_split is None:
            return {'class': np.bincount(y).argmax()}

        # Recursively build left and right subtrees
        left_tree = self._build_tree(X[best_split['left_mask']], y[best_split['left_mask']], depth + 1)
        right_tree = self._build_tree(X[best_split['right_mask']], y[best_split['right_mask']], depth + 1)

        return {
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'left_tree': left_tree,
            'right_tree': right_tree
        }

    def _information_gain(self, parent, left, right):
        """
        Calculate information gain of a split.
        """
        parent_entropy = self._entropy(parent)
        left_entropy = self._entropy(left)
        right_entropy = self._entropy(right)

        weighted_avg = (len(left)/len(parent)) * left_entropy + (len(right)/len(parent)) * right_entropy
        return parent_entropy - weighted_avg

    def _entropy(self, y):
        """
        Calculate entropy of a label set.
        """
        if len(y) == 0:
            return 0
        probs = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p + 1e-9) for p in probs if p > 0])

    def predict(self, X):
        """
        Predict class labels for a dataset.
        """
        return np.array([self._predict_single(x, self.tree) for x in X])

    def _predict_single(self, x, tree):
        """
        Predict class label for a single sample.
        """
        if 'class' in tree:
            return tree['class']

        feature_val = x[tree['feature_idx']]
        if feature_val <= tree['threshold']:
            return self._predict_single(x, tree['left_tree'])
        else:
            return self._predict_single(x, tree['right_tree'])


Step -2- Load and Split the Iris Datasets:

In [73]:
# Load Iris Dataset and Split into Train/Test

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


Training samples: 120
Test samples: 30


Step -3- Train and Evaluate a Custom Decision Tree:

In [74]:
# Train and Evaluate Custom Decision Tree

from sklearn.metrics import accuracy_score

# Initialize custom decision tree with max_depth=3
custom_tree = CustomDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)

# Predict on test data
y_pred_custom = custom_tree.predict(X_test)

# Calculate accuracy
accuracy_custom = accuracy_score(y_test, y_pred_custom)
print(f"Custom Decision Tree Accuracy: {accuracy_custom:.4f}")


Custom Decision Tree Accuracy: 1.0000


Step -4- Train and Evaluate a Scikit Learn Decision Tree:

In [75]:
# Train and Evaluate Scikit-learn Decision Tree

from sklearn.tree import DecisionTreeClassifier

# Initialize scikit-learn decision tree
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)

# Predict on test data
y_pred_sklearn = sklearn_tree.predict(X_test)

# Calculate accuracy
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f"Scikit-learn Decision Tree Accuracy: {accuracy_sklearn:.4f}")

# Compare results
print("\nAccuracy Comparison:")
print(f"Custom Decision Tree: {accuracy_custom:.4f}")
print(f"Scikit-learn Decision Tree: {accuracy_sklearn:.4f}")


Scikit-learn Decision Tree Accuracy: 1.0000

Accuracy Comparison:
Custom Decision Tree: 1.0000
Scikit-learn Decision Tree: 1.0000


Step -5- Result Comparison:

In [76]:
# Classification Models: Wine Dataset
# Decision Tree & Random Forest

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

wine = load_wine()
X_w, y_w = wine.data, wine.target

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

# Decision Tree
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train_w, y_train_w)
y_pred_dt = dt_classifier.predict(X_test_w)
f1_dt = f1_score(y_test_w, y_pred_dt, average='weighted')
print(f"Decision Tree F1 Score: {f1_dt:.4f}")

# Random Forest
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_w, y_train_w)
y_pred_rf = rf_classifier.predict(X_test_w)
f1_rf = f1_score(y_test_w, y_pred_rf, average='weighted')
print(f"Random Forest F1 Score: {f1_rf:.4f}")




Decision Tree F1 Score: 0.9440
Random Forest F1 Score: 1.0000


**3 Exercise - Ensemble Methods and Hyperparameter Tuning.**

Hyperparameter Tuning:

In [82]:
#Implement Classification Models:
wine = load_wine()
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)

print("Decision Tree F1 Score:", f1_score(y_test, y_pred_dt, average="weighted"))
print("Random Forest F1 Score:", f1_score(y_test, y_pred_rf, average="weighted"))

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Decision Tree Regressor
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)

# Random Forest Regressor
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train, y_train)

Decision Tree F1 Score: 0.9439974457215836
Random Forest F1 Score: 1.0


RandomForestRegressor(random_state=42)

In [78]:

#Hyperparameter Tuning (Random Forest – GridSearchCV)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best F1 Score: 0.9782952128219708


In [79]:

rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train, y_train)

rf_reg_preds = rf_reg.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_reg_preds)

print(f"Random Forest Regressor MSE: {rf_mse:.4f}")



Decision Tree MSE: 0.495235205629094
Random Forest MSE: 0.2553684927247781


In [80]:
param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    scoring="neg_mean_squared_error",
    cv=5,
    random_state=42
)

random_search.fit(X_train, y_train)

Best Parameters: {'n_estimators': 100, 'max_features': 'log2', 'max_depth': 20}
Best MSE: 0.2458085852355155


In [81]:
print("Best Hyperparameters (Regression):")
print(random_search.best_params_)

print("\nBest Score (Negative MSE):")
print(random_search.best_score_)


Best Hyperparameters (Regression):
{'n_estimators': 100, 'max_features': 'log2', 'max_depth': 20}

Best Score (Negative MSE):
-0.2458085852355155
